### Agents as graphs (the LangGraph way)

LangGraph was inspired by other graph-based tools to apply a graph-first mode of thinking to agents.

Graph **nodes** and **edges** allow agent developers to map out the exact components and actions to build more complex workflows involving LLMs.

To get started, you'll build a graph state, which will save user and assistant messages in a conversation history, so conversations can progress.

To do this, we first create a `State` class:

```py
class State(TypedDict):
    messages: Annotated[list, add_messages]
```

All this means is that the `State` class has some messages (`messages`), and these messages are a list where new messages are appended to it rather than overridden (indicated with the `add_messages()` function).

This `State` class is then converted into a graph state object using `StateGraph`.

In [ ]:
%pip install --quiet langgraph==0.5.3 langchain-openai==0.3.16 pydantic==2.11.9

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

#Create the state tp ca[ture messages
class State(TypedDict):
    messages: Annotated[list, add_messages]
    
#Create the graph state
graph_builder = StateGraph(State)

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Load variables from the .env file
load_dotenv()

# Define the model using OpenRouter
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Takes the state, and appends the new messages to it
def llm_node(state):
    return {"messages": [llm.invoke(state["messages"])]}


In [ ]:
#Create a node called "llm" that calss the llm_node()function
graph_builder.add_node("llm", llm_node)

#Connect the "llm" node to the START and END of the graph
graph_builder.add_edge(START, "llm")
graph_builder.add_edge("llm", END)

#Compile the graph
graph = graph_builder.compile()

In [ ]:
# Visualize  graph
graph

In [ ]:
from course_helper_functions import pretty_print_messages

for chunk in graph.stream(
    {"messages": [{"role": "user", "content": "Tell me about Apple Inc."}]}
):
    for node, update in chunk.items():
        print(f"\nUpdate from node {node}:\n")
        if "messages" in update:
            pretty_print_messages(update["messages"])